# Long-Term Memory: Searching the Infinite Past

Once an agent has months of conversations, we cannot pass the whole history to the LLM. This notebook explores how agents "recall" only the most relevant facts from a massive archive of data.

---

## 1. The Challenge of Infinite History
LLMs have a **Context Window** limit. Sending everything for every request is expensive and slow. Let's estimate how many "tokens" a conversation might use.

In [10]:
def estimate_tokens(text):
    # Rough estimation: 1 token ≈ 4 characters
    return len(text) // 4

conversation = [
    "Hi, how are you?", "I'm great! How can I help?",
    "What is the weather like in Tokyo?", "It's sunny in Tokyo today."
] * 100 # Simulating a long chat

total_text = " ".join(conversation)
print(f"Estimated Tokens: {estimate_tokens(total_text)}")
print("As context grows, we MUST move old data to Long-Term Memory.")

Estimated Tokens: 2649
As context grows, we MUST move old data to Long-Term Memory.


## 2. Manual Archiving
The first step is moving "cold" data out of the active prompt. We store old messages in an 'Archive'.

In [11]:
history = [f"Message {i}" for i in range(1, 21)]

def archive_cycle(active_history, threshold=5):
    archive = active_history[:-threshold]
    live = active_history[-threshold:]
    return live, archive

live, cold_storage = archive_cycle(history)
print(f"Live (Active): {live}")
print(f"Cold Storage (Archived): {len(cold_storage)} messages")

Live (Active): ['Message 16', 'Message 17', 'Message 18', 'Message 19', 'Message 20']
Cold Storage (Archived): 15 messages


## 3. Keyword-Based Retrieval
We can search the archive by specific words. We'll add a simple "Frequency Score" to see which archive entry is most relevant.

In [12]:
archive_data = [
    "The user loves Python and AI.",
    "The user owns a golden retriever dog.",
    "The user is from Chennai, India."
]

def search_archive(query, data):
    query_words = set(query.lower().split())
    results = []
    for entry in data:
        entry_words = set(entry.lower().split())
        score = len(query_words.intersection(entry_words))
        if score > 0:
            results.append((score, entry))
    return sorted(results, reverse=True)

print(f"Search Results for 'python': {search_archive('I want python code', archive_data)}")

Search Results for 'python': [(1, 'The user loves Python and AI.')]


## 4. Introduction to Semantic Memory
Keyword search misses synonyms. **Embeddings** solve this by converting text into coordinates. Let's visualize a mock 3D vector for a word.

In [13]:
import random

def mock_embedding(text):
    # In reality, Gemini creates a 1536-dimensional vector
    # We will use 3 dimensions for visualization
    return [random.uniform(-1, 1) for _ in range(3)]

print(f"Embedding for 'Dog': {mock_embedding('Dog')}")
print(f"Embedding for 'Pet': {mock_embedding('Pet')}")
print("Semantic memory finds meanings that are physically 'close' in this coordinate space.")

Embedding for 'Dog': [0.6703583821620698, 0.7367509516038049, 0.8615445328449363]
Embedding for 'Pet': [-0.16942053338398044, 0.7957078213132858, -0.14266482464724217]
Semantic memory finds meanings that are physically 'close' in this coordinate space.


## 5. What is a Vector Database?
Let's build a `SimpleVectorStore` that calculates the distance between meanings.

In [14]:
import math

class SimpleVectorStore:
    def __init__(self):
        self.store = {}

    def add(self, text, vector):
        self.store[text] = vector

    def search(self, query_vector):
        # Calculate Euclidean distance: lower is better
        results = []
        for text, vec in self.store.items():
            dist = math.sqrt(sum([(a-b)**2 for a, b in zip(query_vector, vec)]))
            results.append((dist, text))
        return sorted(results)

vdb = SimpleVectorStore()
vdb.add("I like apples", [0.1, 0.8, -0.1])
vdb.add("The sky is blue", [-0.8, -0.2, 0.9])

print(f"Top Result: {vdb.search([0.0, 0.7, 0.0])[0][1]}")

Top Result: I like apples


## 6. The RAG Pattern
**RAG** (Retrieval Augmented Generation) is the loop of Query -> Search -> Augment -> Answer.

In [15]:
def rag_workflow(query):
    print("1. Receive Query: ", query)
    print("2. Search Vector DB for relevant context...")
    context = "User's pet name is Bruno."
    print("3. Augment Prompt with retrieved context.")
    final_prompt = f"Use this: {context}. Question: {query}"
    print("4. LLM Generates Final Answer.")
    return final_prompt # In reality, we call the LLM here

print(rag_workflow("What is my pet's name?"))

1. Receive Query:  What is my pet's name?
2. Search Vector DB for relevant context...
3. Augment Prompt with retrieved context.
4. LLM Generates Final Answer.
Use this: User's pet name is Bruno.. Question: What is my pet's name?


## 7. Implementation: Full RAG Orchestrator
Let's use our `gemini-2.5-flash` model to answer a question based on a retrieval store.

In [16]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()
# Note: gemini-2.5-flash is used as requested
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

def ask_agent_with_long_term_memory(query, knowledge_base):
    # Retrieval step
    related = [m for m in knowledge_base if query.lower() in m.lower()]
    
    prompt = f"""You are an assistant with access to a library.
    Library Facts: {related}
    Question: {query}
    """
    return llm.invoke(prompt).content

library = ["The project deadline is April 30.", "The team lead is Bala."]
print(ask_agent_with_long_term_memory("When is the deadline?", library))

I need a bit more information to help you with that!

What deadline are you referring to? For example, are you asking about:

*   A book return date?
*   A library program registration?
*   An interlibrary loan?
*   A fine payment?

Once I know what you're asking about, I can try to find the specific deadline for you.


## 8. Entity Extraction Mockup
Advanced agents store facts as structured data. Let's mock an extraction.

In [17]:
def extract_entities(text):
    # Mocking what an LLM would do
    entities = {"location": "Unknown", "action": "Unknown"}
    if "paris" in text.lower(): entities["location"] = "Paris"
    if "running" in text.lower(): entities["action"] = "Running"
    return entities

print(f"Extracted Data: {extract_entities('The user is running in Paris.')}")

Extracted Data: {'location': 'Paris', 'action': 'Running'}


## 9. Hybrid Memory Manager
Let's combine Short-term and Long-term into one class.

In [18]:
class HybridMemory:
    def __init__(self):
        self.short_term = [] # Last 5 messages
        self.long_term = [] # Full Archive
    
    def add_message(self, msg):
        self.short_term.append(msg)
        self.long_term.append(msg)
        if len(self.short_term) > 5: self.short_term.pop(0)

memory_system = HybridMemory()
memory_system.add_message("User: Hi")
print(f"Short term state: {memory_system.short_term}")

Short term state: ['User: Hi']


## 10. Summary & Pro-Tips
- **Filter then Search**: Reduce noise before doing vector searches.
- **Data Privacy**: Encrypt your long-term storage.
- **RAG is Key**: Master RAG to build truly powerful agents.

---